<a href="https://colab.research.google.com/github/HariniRJ/ai-lab/blob/main/WEEK_2_DFS_(IDDFS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import time

def get_moves(state):
    """Generates all valid next states from the current board state."""
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)
    moves = []

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col

            state_list = list(state)
            state_list[zero_idx], state_list[new_idx] = (
                state_list[new_idx],
                state_list[zero_idx]
            )

            moves.append(tuple(state_list))

    return moves


def depth_limited_dfs(current, goal, limit, path, visited, nodes_explored):
    """Recursive depth-limited DFS."""
    nodes_explored[0] += 1

    if current == goal:
        return path

    if limit <= 0:
        return None

    visited.add(current)

    for next_state in get_moves(current):
        if next_state not in visited:
            result = depth_limited_dfs(
                next_state,
                goal,
                limit - 1,
                path + [next_state],
                visited,
                nodes_explored
            )

            if result is not None:
                return result

    visited.remove(current)
    return None


def iddfs(start, goal, max_depth_limit=None):
    """Main IDDFS loop."""
    depth = 0
    total_nodes_explored = [0]

    while True:
        print(f"Searching at depth limit: {depth}...")

        visited = set()
        nodes_explored_this_depth = [0]

        result = depth_limited_dfs(
            start,
            goal,
            depth,
            [start],
            visited,
            nodes_explored_this_depth
        )

        total_nodes_explored[0] += nodes_explored_this_depth[0]

        if result is not None:
            return result, depth, total_nodes_explored[0]

        depth += 1

        if max_depth_limit is not None and depth > max_depth_limit:
            print(
                f"Max depth limit of {max_depth_limit} reached. "
                "No solution found within this limit."
            )
            return None, max_depth_limit, total_nodes_explored[0]


def print_board(state):
    """Prints the board in 3x3 format."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]}")
    print()


def get_user_input_board(prompt_name):
    """Gets and validates the puzzle board."""
    while True:
        print(f"\nEnter the {prompt_name} state row by row.")
        print("Provide 3 numbers per row separated by spaces.")
        print("Use '0' for the blank space.")

        digits = []
        error_occurred = False

        for r in range(3):
            try:
                row_str = input(f"Row {r+1}: ").strip()
                row_digits = [int(x) for x in row_str.split()]

                if len(row_digits) != 3:
                    print("Error: Each row must contain exactly 3 numbers.")
                    error_occurred = True
                    break

                digits.extend(row_digits)

            except ValueError:
                print("Error: Enter numbers only.")
                error_occurred = True
                break

        if error_occurred:
            continue

        if len(digits) != 9 or set(digits) != set(range(9)):
            print("Error: Enter numbers from 0 to 8 without duplicates.")
            continue

        return tuple(digits)


def get_user_input_int(prompt):
    """Gets a valid non-negative integer."""
    while True:
        try:
            value = int(input(prompt).strip())

            if value < 0:
                print("Please enter a non-negative integer.")
            else:
                return value

        except ValueError:
            print("Invalid input. Enter a whole number.")


# ---------------- MAIN PROGRAM ----------------

if __name__ == "__main__":

    print("=== 8-Puzzle Solver using Iterative Deepening DFS (IDDFS) ===")
    print("Name: H G Shreyas")
    print("USN: 1BM24CS108")

    start_state = get_user_input_board("STARTING")
    goal_state = get_user_input_board("GOAL")

    max_depth_limit = get_user_input_int(
        "\nEnter a maximum depth limit for the search "
        "(e.g., 20, 0 for no limit): "
    )

    if max_depth_limit == 0:
        max_depth_limit = None

    print("\n--- Initializing Search ---")

    print("Initial State:")
    print_board(start_state)

    print("Goal State:")
    print_board(goal_state)

    if max_depth_limit is not None:
        print(f"Max Depth Limit: {max_depth_limit}")

    start_timer = time.time()

    solution_path, final_depth, explored_nodes_count = iddfs(
        start_state,
        goal_state,
        max_depth_limit
    )

    end_timer = time.time()
    execution_time = end_timer - start_timer

    if solution_path is not None:

        print(f"\nSuccess! Optimal solution found at depth: {final_depth}")
        print(f"Total moves required: {len(solution_path) - 1}")
        print(f"Total nodes explored: {explored_nodes_count}")

        print("\n--- Step-by-Step Path ---")

        for step, state in enumerate(solution_path):

            if step == 0:
                print(f"Step {step} (Start):")
            else:
                print(f"Step {step}:")

            print_board(state)

        # Empirical Performance
        print("\n--- Empirical Performance (IDDFS) ---")

        print(
            "Total Explored States (Nodes processed across all "
            f"DLS iterations): {explored_nodes_count}"
        )

        print(f"Execution Time: {execution_time:.4f} seconds")

        print(
            f"Max Stack Depth for a DLS iteration (path length): "
            f"{final_depth + 1} nodes"
        )

        # Theoretical Complexities
        branching_factor = 3

        theoretical_time_nodes_approx = branching_factor ** final_depth
        theoretical_space_nodes = branching_factor ** final_depth

        print("\n--- Theoretical Complexities ---")

        print(
            f"For Solution Depth (d) = {final_depth}, "
            f"Branching Factor (b) = {branching_factor}"
        )

        print(
            "Theoretical Time Complexity (approximated O(b^d)): "
            f"{int(theoretical_time_nodes_approx)} nodes"
        )

        print(
            "Theoretical Space Complexity (O(b*d)): "
            f"{int(theoretical_space_nodes)} nodes"
        )

    else:

        print(
            f"\nNo solution found within the specified maximum "
            f"depth limit of {max_depth_limit}."
        )

        print(f"Total nodes explored until limit: {explored_nodes_count}")

        print("\n--- Empirical Performance (IDDFS until limit) ---")

        print(
            "Total Explored States (Nodes processed across all "
            f"DLS iterations): {explored_nodes_count}"
        )

        print(f"Execution Time: {execution_time:.4f} seconds")

        if max_depth_limit is not None:
            print(
                f"Max Stack Depth for a DLS iteration (path length): "
                f"{max_depth_limit + 1} nodes"
            )
        else:
            print(
                "Max Stack Depth for a DLS iteration (path length): "
                "Unknown (no limit specified)"
            )

    # General Complexity Analysis
    print("\n--- General Complexity Analysis for IDDFS ---")

    print("Time Complexity: O(b^d)")
    print(
        "'b' is the branching factor (average number of possible moves)."
    )
    print(
        "'d' is the depth of the shallowest goal state. "
        "IDDFS explores states at shallower depths multiple times."
    )

    print("Space Complexity: O(b*d)")
    print(
        "IDDFS performs Depth-First Search at each iteration, "
        "so it stores the current search path."
    )


=== 8-Puzzle Solver using Iterative Deepening DFS (IDDFS) ===
Name: H G Shreyas
USN: 1BM24CS108

Enter the STARTING state row by row.
Provide 3 numbers per row separated by spaces.
Use '0' for the blank space.
Row 1: 1 2 3
Row 2: 4 5 6
Row 3: 7 0 8

Enter the GOAL state row by row.
Provide 3 numbers per row separated by spaces.
Use '0' for the blank space.
Row 1: 1 2 3
Row 2: 4 5 6 
Row 3: 7 8 0

Enter a maximum depth limit for the search (e.g., 20, 0 for no limit): 20

--- Initializing Search ---
Initial State:
 1 2 3
 4 5 6
 7 0 8

Goal State:
 1 2 3
 4 5 6
 7 8 0

Max Depth Limit: 20
Searching at depth limit: 0...
Searching at depth limit: 1...

Success! Optimal solution found at depth: 1
Total moves required: 1
Total nodes explored: 5

--- Step-by-Step Path ---
Step 0 (Start):
 1 2 3
 4 5 6
 7 0 8

Step 1:
 1 2 3
 4 5 6
 7 8 0


--- Empirical Performance (IDDFS) ---
Total Explored States (Nodes processed across all DLS iterations): 5
Execution Time: 0.0000 seconds
Max Stack Depth f